# Retail Customer Sales Analysis – Exploratory Data Analysis (EDA)

**Student:** Navya Mishra  
**Roll Number:** 2026M01410094  
**Programme:** MCA  
**Project:** Project 1 – Exploratory Data Analysis  
**Dataset:** `retail_customer_sales_dataset.csv`

## Project purpose
This notebook follows the supplied Project 1 brief: audit the dataset, justify cleaning decisions, calculate descriptive statistics, examine numerical and categorical variables, review outliers, create at least three calculated fields, and answer business questions with grouped summaries.

**How to use:** Keep the CSV in the same folder as this notebook. Run cells in order. Read the markdown explanations, check the output, and be prepared to explain each cleaning decision in your own words.

## 1. Import libraries and load the dataset
Required libraries: pandas, NumPy, Matplotlib and Seaborn. The charts below use Matplotlib for straightforward, reproducible plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
sns.set_theme(style="whitegrid")

data_file = Path("retail_customer_sales_dataset.csv")
df = pd.read_csv(data_file)

print("Dataset shape:", df.shape)
display(df.head())
display(df.tail())


: 

## 2. First look: columns, data types and structure

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nData types:")
display(df.dtypes.to_frame("Data type"))

print("\\nDataset information:")
df.info()

print("\\nNumerical summary:")
display(df.describe().T)

## 3. Data-quality audit
The audit records missing counts and percentages, exact duplicate rows, duplicate transaction identifiers, category variants, numerical values outside expected ranges, and date-order problems.

In [ ]:
missing_audit = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percent": (df.isna().sum() / len(df) * 100).round(2)
}).sort_values("Missing Count", ascending=False)
display(missing_audit)

print("Exact duplicate rows:", df.duplicated().sum())
print("Repeated Transaction_ID values:", df["Transaction_ID"].duplicated().sum())

for column in ["Gender", "Region", "Customer_Segment", "Product_Category",
               "Payment_Method", "Delivery_Mode", "Satisfaction_Level", "Returned"]:
    print(f"\\nValues in {column}:")
    print(df[column].value_counts(dropna=False).to_string())

print("\\nAge outside 18–100:", (~df["Age"].between(18, 100)).sum())
print("Discount outside 0–0.50:", (~df["Discount"].between(0, 0.50)).sum())
print("Income values at or below zero:", (df["Income"] <= 0).sum())

order_dates = pd.to_datetime(df["Order_Date"], format="mixed", errors="coerce", dayfirst=True)
delivery_dates = pd.to_datetime(df["Delivery_Date"], format="mixed", errors="coerce", dayfirst=True)
print("Order dates that could not be parsed:", order_dates.isna().sum())
print("Delivery dates that could not be parsed:", delivery_dates.isna().sum())
print("Delivery dates earlier than order dates:", (delivery_dates < order_dates).sum())

## 4. Cleaning decisions and reasons

- Remove exact duplicate records because they repeat a complete transaction row.
- Standardize text by trimming spaces and using consistent title case. Map `M`/`F` to `Male`/`Female`, and the observed region variants `Sth`/`Nort` to `South`/`North`.
- Parse dates with mixed-format handling. If a delivery date is earlier than the order date, set it to missing because the sequence is not logically valid.
- Treat ages outside 18–100 and discounts outside 0–0.50 as invalid. Replace invalid ages with missing, then fill missing Age and Income with their medians. This preserves rows while reducing sensitivity to extreme values.
- Keep missing Customer_Rating and Discount values missing rather than inventing values. Mark missing Satisfaction_Level as `Unknown`.
- Keep valid extreme Income, Sales and Profit observations; flag them for review rather than deleting them automatically.

These are explicit project choices. In a real company, they should be confirmed with the data owner.

In [ ]:
clean_df = df.drop_duplicates().copy()

# Standardize categorical labels
category_columns = ["Gender", "Region", "Customer_Segment", "Product_Category",
                    "Payment_Method", "Delivery_Mode", "Satisfaction_Level", "Returned"]
for column in category_columns:
    clean_df[column] = clean_df[column].astype("string").str.strip().str.title()

clean_df["Gender"] = clean_df["Gender"].replace({"M": "Male", "F": "Female"})
clean_df["Region"] = clean_df["Region"].replace({"Sth": "South", "Nort": "North"})

# Parse mixed date formats
clean_df["Order_Date"] = pd.to_datetime(
    clean_df["Order_Date"], format="mixed", errors="coerce", dayfirst=True
)
clean_df["Delivery_Date"] = pd.to_datetime(
    clean_df["Delivery_Date"], format="mixed", errors="coerce", dayfirst=True
)

# Mark impossible values as missing
clean_df.loc[~clean_df["Age"].between(18, 100), "Age"] = np.nan
clean_df.loc[~clean_df["Discount"].between(0, 0.50), "Discount"] = np.nan
clean_df.loc[clean_df["Income"] <= 0, "Income"] = np.nan

# Fill selected numerical fields using their medians
clean_df["Age"] = clean_df["Age"].fillna(clean_df["Age"].median())
clean_df["Income"] = clean_df["Income"].fillna(clean_df["Income"].median())

# Do not invent ratings or discounts
clean_df["Satisfaction_Level"] = clean_df["Satisfaction_Level"].fillna("Unknown")

# Delivery before order is invalid
wrong_delivery = clean_df["Delivery_Date"] < clean_df["Order_Date"]
clean_df.loc[wrong_delivery, "Delivery_Date"] = pd.NaT

# Standardization can make formerly different text spellings identical.
duplicates_after_standardizing = clean_df.duplicated().sum()
clean_df = clean_df.drop_duplicates().copy()

print("Rows in original data:", len(df))
print("Rows after cleaning:", len(clean_df))
print("Exact duplicates removed initially:", df.duplicated().sum())
print("Additional duplicates after standardizing labels:", duplicates_after_standardizing)
print("Duplicates remaining:", clean_df.duplicated().sum())
display(clean_df.isna().sum().to_frame("Missing values after cleaning"))

## 5. Descriptive statistics
The project brief asks for mean, median, mode, standard deviation, variance, quartiles, percentiles and IQR for Age, Income, Quantity, Unit_Price, Discount, Sales, Cost and Profit.

In [ ]:
numeric_columns = ["Age", "Income", "Quantity", "Unit_Price",
                   "Discount", "Sales", "Cost", "Profit"]

descriptive = clean_df[numeric_columns].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]).T
descriptive["Median"] = clean_df[numeric_columns].median()
descriptive["Mode"] = clean_df[numeric_columns].mode().iloc[0]
descriptive["Variance"] = clean_df[numeric_columns].var()
descriptive["IQR"] = descriptive["75%"] - descriptive["25%"]
descriptive["Skewness"] = clean_df[numeric_columns].skew()
display(descriptive.round(2))

## 6. Univariate analysis: numerical variables
Histograms show the shape of at least four numerical variables. Skewness values help describe whether the distribution is symmetric or has a longer tail.

In [ ]:
for column in ["Age", "Income", "Sales", "Profit"]:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(clean_df[column].dropna(), bins=30, edgecolor="black")
    ax.set_title(f"Distribution of {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Number of transactions")
    fig.tight_layout()
    plt.show()

display(clean_df[["Age", "Income", "Sales", "Profit"]].skew().to_frame("Skewness"))

## 7. Univariate analysis: categorical variables
Frequency and proportion tables are provided for Region, Product_Category and Customer_Segment.

In [ ]:
for column in ["Region", "Product_Category", "Customer_Segment"]:
    frequency = clean_df[column].value_counts(dropna=False)
    proportion = clean_df[column].value_counts(normalize=True, dropna=False).mul(100)
    table = pd.DataFrame({"Frequency": frequency, "Percentage": proportion.round(2)})
    display(table)

    fig, ax = plt.subplots(figsize=(8, 4))
    frequency.plot(kind="bar", ax=ax, edgecolor="black")
    ax.set_title(f"Frequency of {column}")
    ax.set_xlabel(column.replace("_", " "))
    ax.set_ylabel("Number of transactions")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    plt.show()

## 8. Outlier detection using IQR
For each variable, values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR are flagged. A flag is not automatically a data error. Large sales or income values can be genuine.

In [ ]:
outlier_rows = []
for column in ["Sales", "Profit", "Age", "Income"]:
    q1 = clean_df[column].quantile(0.25)
    q3 = clean_df[column].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    flagged = ((clean_df[column] < lower_fence) |
               (clean_df[column] > upper_fence)).sum()
    outlier_rows.append({
        "Variable": column,
        "Lower fence": lower_fence,
        "Upper fence": upper_fence,
        "Flagged rows": int(flagged)
    })

outlier_summary = pd.DataFrame(outlier_rows)
display(outlier_summary.round(2))

for column in ["Sales", "Profit", "Age", "Income"]:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.boxplot(clean_df[column].dropna(), vert=False)
    ax.set_title(f"Outlier review for {column}")
    ax.set_xlabel(column)
    fig.tight_layout()
    plt.show()

## 9. Feature engineering
Three calculated columns are created using the supplied brief's examples: Profit_Margin, Delivery_Days and Revenue_per_Customer.

In [ ]:
clean_df["Profit_Margin"] = np.where(
    clean_df["Sales"] != 0,
    clean_df["Profit"] / clean_df["Sales"] * 100,
    np.nan
)

clean_df["Delivery_Days"] = (
    clean_df["Delivery_Date"] - clean_df["Order_Date"]
).dt.days

clean_df["Revenue_per_Customer"] = (
    clean_df.groupby("Customer_ID")["Sales"].transform("sum")
)

display(clean_df[[
    "Transaction_ID", "Sales", "Profit", "Profit_Margin",
    "Delivery_Days", "Revenue_per_Customer"
]].head())

## 10. Business questions using groupby and aggregation
The summaries below answer more than five business questions. Use the results to explain the business implication, not just repeat the largest number.

In [ ]:
# Q1. Which region has the highest total sales?
sales_by_region = clean_df.groupby("Region")["Sales"].agg(
    Total_Sales="sum", Average_Sales="mean", Transactions="count"
).sort_values("Total_Sales", ascending=False)
display(sales_by_region)

# Q2. Which product category generates the highest total profit?
profit_by_category = clean_df.groupby("Product_Category")["Profit"].sum().sort_values(ascending=False)
display(profit_by_category)

# Q3. Which product category has the lowest profit margin?
category_margin = clean_df.groupby("Product_Category").agg(
    Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum")
)
category_margin["Profit_Margin_Percent"] = (
    category_margin["Total_Profit"] / category_margin["Total_Sales"] * 100
)
display(category_margin.sort_values("Profit_Margin_Percent"))

# Q4. Which customer segment has the highest average transaction value?
sales_by_segment = clean_df.groupby("Customer_Segment")["Sales"].agg(
    Average_Sales="mean", Median_Sales="median", Transactions="count"
).sort_values("Average_Sales", ascending=False)
display(sales_by_segment)

# Q5. Which payment method is used most often?
display(clean_df["Payment_Method"].value_counts().to_frame("Transactions"))

# Q6. Which delivery mode has the shortest average delivery time?
delivery_by_mode = clean_df.groupby("Delivery_Mode")["Delivery_Days"].agg(
    Average_Days="mean", Median_Days="median", Valid_Deliveries="count"
).sort_values("Average_Days")
display(delivery_by_mode)

# Q7. Which product category has the highest return rate?
return_rate = clean_df.assign(
    Returned_Flag=clean_df["Returned"].str.lower().eq("yes")
).groupby("Product_Category")["Returned_Flag"].mean().mul(100).sort_values(ascending=False)
display(return_rate.to_frame("Return_Rate_Percent"))

## 11. Save the cleaned dataset
The output CSV is saved next to this notebook so it can be included in the submission.

In [ ]:
clean_df.to_csv("2026M01410094_Cleaned_Retail_Dataset.csv", index=False)
print("Cleaned dataset saved.")
print("Final shape:", clean_df.shape)
print("Duplicate rows remaining:", clean_df.duplicated().sum())

## 12. Final review checklist

- [ ] I ran every notebook cell from top to bottom.
- [ ] I checked the audit counts and cleaning output.
- [ ] I can explain why median imputation was used for Age and Income.
- [ ] I can explain why missing ratings and discounts were not invented.
- [ ] I can explain why outliers were flagged instead of automatically deleted.
- [ ] I reviewed the groupby results and can explain the business implications.
- [ ] I have the cleaned CSV and the written insights report.
- [ ] My roll number is correct on all submitted files.

**Academic integrity:** This notebook is a structured working draft based on the supplied project brief and dataset. Review and understand the code, verify the outputs on your own Jupyter installation, and personalize the business interpretation in your own words. Cite any external code or tutorial you adapt.